# 2. Missingness

Continues from [0_load_and_orient.ipynb](0_load_and_orient.ipynb) — reloads
the same setup so this notebook runs standalone.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# load csv data for both train and test
df_train = pd.read_csv("../../data/train.csv")
df_test = pd.read_csv("../../data/test.csv")

In [ ]:
# missing counts and percent per column for train data
missing_total = df_train.isnull().sum().sort_values(ascending=False)
missing_total_percent = (df_train.isnull().sum() / df_train.isnull().count()).sort_values(ascending=False) * 100
missing_data = pd.concat([missing_total, missing_total_percent], keys=['missing_total', 'missing_total_percent'], axis=1)
missing_data.head(20)

In [ ]:
# filter total > 0
missing_data = missing_data[missing_data["missing_total"] > 0]
missing_data

In [ ]:
len(missing_data)

In [ ]:
# missing counts and percent per column for test data
missing_total_test = df_test.isnull().sum().sort_values(ascending=False)
missing_total_percent_test = (df_test.isnull().sum() / df_test.isnull().count()).sort_values(ascending=False) * 100
missing_data_test = pd.concat([missing_total_test, missing_total_percent_test], keys=['missing_total_test', 'missing_total_percent_test'], axis=1)
missing_data_test.head(20)

In [ ]:
len(missing_data_test)

In [ ]:
# filter total > 0 for missing test data
missing_data_test = missing_data_test[missing_data_test["missing_total_test"] > 0]
missing_data_test

In [ ]:
len(missing_data_test)

In [ ]:
# verify GarageType NaN really means "no garage"
df_train[df_train["GarageType"].isnull()]

In [ ]:
df_train[df_train["GarageType"].isnull()][["GarageType", "GarageCars", "GarageArea"]].describe()

since max for GarageCars and GarageArea both are 0's. So, GarageType -> NaN means structural absence not unknown data.

In [ ]:
# fill Nan with 'None'
df_train["GarageType"] = df_train["GarageType"].fillna("None")
df_test["GarageType"] = df_test["GarageType"].fillna("None")

In [ ]:
df_train[df_train["GarageType"] == "None"][["GarageType"]]

In [ ]:
df_train["GarageType"].value_counts()

#### Bsmt group

In [ ]:
df_train[df_train["BsmtQual"].isnull()][["BsmtQual", "TotalBsmtSF", "BsmtFinSF1", "BsmtUnfSF"]].describe()

Same pattern as 'GarageType', it's structural absence not unknown value.

In [ ]:
# fill NaN with 'None' for the Bsmt group (structural absence, same as GarageType)
bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
for col in bsmt_cols:
    df_train[col] = df_train[col].fillna("None")
    df_test[col] = df_test[col].fillna("None")

In [ ]:
# verify — no more NaNs in the Bsmt group for either split
df_train[bsmt_cols].isnull().sum(), df_test[bsmt_cols].isnull().sum()

In [ ]:
df_train[bsmt_cols].value_counts()

#### Remaining structural-absence group

In [ ]:
# verify PoolQC NaN really means "no pool"
df_train[df_train["PoolQC"].isnull()][["PoolArea"]].describe()

In [ ]:
# verify FireplaceQu NaN really means "no fireplace"
df_train[df_train["FireplaceQu"].isnull()][["Fireplaces"]].describe()

In [ ]:
# verify MasVnrType NaN really means "no masonry veneer"
df_train[df_train["MasVnrType"].isnull()][["MasVnrArea"]].describe()

In [ ]:
# how many MasVnrType-NaN rows actually have a nonzero MasVnrArea?
mask = df_train["MasVnrType"].isnull() & (df_train["MasVnrArea"] > 0)
mask.sum(), df_train[mask][["MasVnrType", "MasVnrArea"]]

`PoolQC` and `FireplaceQu` check out (companion column is exactly 0 for every NaN row). `MasVnrType` doesn't — 5 rows have nonzero `MasVnrArea`, so it's excluded from this group and handled separately. `Alley`, `Fence`, `MiscFeature` have no companion numeric column; treated as structural absence per the data dictionary's stated `NA` meaning.

In [ ]:
# fill NaN with 'None' for the remaining confirmed structural-absence group
structural_absence_cols = ["PoolQC", "FireplaceQu", "Alley", "Fence", "MiscFeature"]
for col in structural_absence_cols:
    df_train[col] = df_train[col].fillna("None")
    df_test[col] = df_test[col].fillna("None")

In [ ]:
# verify — no more NaNs in this group for either split
df_train[structural_absence_cols].isnull().sum(), df_test[structural_absence_cols].isnull().sum()